# Real multilingual relatedness: complete module-driven workflow

This notebook uses authentic human-rated Amharic, Hausa and Kinyarwanda SemRel2024 news pairs. It calls reusable production modules rather than implementing another model. Outputs are intentionally empty: recorded evidence is read from committed reports, while optional cells create genuinely new runs. No generated sentences, pseudo-labels, translations, encoded code, or automatic test rescoring.

Install once in your environment: `python -m pip install -e ".[dev,notebook]"`. Full encoder extraction/inference needs substantial RAM; use the private cloud CPU runner on an 8 GB host. Read `docs/provenance.md`, the source amendment, and frozen protocol first.

In [ ]:
from pathlib import Path
import sys, json, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Repository:', ROOT)
print('Fresh run ID:', RUN_ID)


## 1. Real data and immutable source checks
Fetching and preparation are opt-in. The official CSV checksums are enforced by the production adapter. Preparation strips test targets and removes unordered pair copies using text only; no test ratings are opened for selection.

In [ ]:
FETCH_REAL_DATA = False
PREPARE_NEW_DATA = False
PROCESSED = ROOT / 'data' / 'processed' / 'semrel-lf-v1'
from src.data.make_dataset import fetch, prepare
if FETCH_REAL_DATA:
    source_receipt = fetch(ROOT / 'data' / 'raw' / 'semrel')
    print('Verified official files:', len(source_receipt['files']))
if PREPARE_NEW_DATA:
    PROCESSED = ROOT / 'data' / 'processed' / ('notebook-' + RUN_ID)
    prepare(ROOT / 'data' / 'raw' / 'semrel', PROCESSED)
print('Processed source available:', PROCESSED.exists())


## 2. Connected sentence groups and source diagnostics
This audit is label-free. A shared sentence does not prove copied pair targets, but it means the official split is not sentence-disjoint. Hausa contains a large connected component. Cluster uncertainty must be interpreted accordingly.

In [ ]:
from src.data.components import audit_components
if PROCESSED.exists():
    splits = {s: [json.loads(line) for line in (PROCESSED / (s + '.jsonl')).read_text(encoding='utf-8').splitlines()] for s in ('train', 'dev', 'test')}
    groups, component_report = audit_components(splits)
    print(json.dumps(component_report, indent=2))
    assert all('target' not in row for row in splits['test'])
else:
    print('Run real-data preparation first; no substitute dataset is created.')


## 3. Pinned frozen encoder features
The backbone is MLM-adapted AfroXLMR, not a sentence-similarity encoder. Mean pooling uses the attention mask, no random pooler; embeddings are unit normalized, max length 128. The pinned revision, loading keys, weight hashes and truncation diagnostics are persisted. Prefer `scripts/stage_kaggle_source.py` + readable `scripts/kaggle_encode.py` in private cloud CPU. Never load the large encoder merely to view saved results.

In [ ]:
RUN_ENCODER_LOCALLY = False
CACHE = ROOT / 'data' / 'features' / 'afro-xlmr-v1'
if RUN_ENCODER_LOCALLY:
    from src.models.encoder import cache_dataset
    CACHE = ROOT / 'data' / 'features' / ('notebook-' + RUN_ID)
    cache_dataset(PROCESSED, CACHE, device='cpu')
receipt_path = CACHE / 'receipt.json'
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    print({k: receipt[k] for k in ('model', 'revision', 'pooling', 'max_tokens', 'splits')})
else:
    print('Authentic encoder cache not present; use the documented private cloud extraction.')


## 4. Actual learned training, only when explicitly requested
This production command trains ridge alphas 1/10/100 and three MLP configurations with seeds 42/1337/2026, language-balanced weighted MSE and train-only normalization. Each seed uses validation checkpoint selection, configurations use equal-seed ensembles. Promotion requires +0.005 macro Spearman and no language loss greater than 0.02. Test gold is never opened by training. Existing run directories are rejected.

In [ ]:
RUN_TRAINING = False
TRAINED = ROOT / 'models' / ('notebook-' + RUN_ID)
if RUN_TRAINING:
    from src.models.train import train
    selection = train(CACHE, PROCESSED, TRAINED)
    print('Validation-selected policy:', selection['selected'])
else:
    print('Training disabled; recorded evidence below is separate from a new run.')


## 5. Recorded validation comparison and learning curves
These cells read aggregate reports only. Missing reports remain missing; no placeholder scores or fake executed outputs are inserted.

In [ ]:
reports = sorted((ROOT / 'docs').glob('selection-v*.json'))
recorded = json.loads(reports[-1].read_text(encoding='utf-8')) if reports else None
if recorded:
    comparison = [(r['name'], r['validation']['macro_spearman']) for r in recorded['candidates']]
    print(comparison)
    print('Selected:', recorded['selected'])
else:
    print('No completed recorded model selection report yet.')


In [ ]:
if recorded:
    import matplotlib.pyplot as plt
    rows = [(r['name'], r['validation']['macro_spearman']) for r in recorded['candidates'] if r['validation']['macro_spearman'] is not None]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar([r[0] for r in rows], [r[1] for r in rows]); ax.tick_params(axis='x', rotation=45)
    ax.set(ylabel='Validation language-macro Spearman', title='Recorded candidates; no test-guided choice')
    plt.tight_layout()
    fig, ax = plt.subplots(figsize=(9, 4))
    for candidate in recorded['candidates']:
        for fit in candidate.get('fits', []):
            ax.plot([r['epoch'] for r in fit['history']], [r['validation']['macro_spearman'] for r in fit['history']], alpha=.6, label=candidate['name'] + '/' + str(fit['seed']))
    ax.set(xlabel='Epoch', ylabel='Validation macro Spearman'); ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()


## 6. Actual saved-model text inference
Opt-in on real development pairs, not invented examples or test cases. The production inference command loads the exact encoder revision/weight identity and weights-only regressor checkpoint. Use cloud CPU if memory is limited. Each output path is fresh. This is inference, not evaluation or a test rerun.

In [ ]:
RUN_TEXT_INFERENCE = False
CHECKPOINT = ROOT / 'models' / 'selected-v1' / 'selected.pt'
if RUN_TEXT_INFERENCE:
    from src.models.predict import predict
    from src.config import LANGUAGES
    dev = [json.loads(line) for line in (PROCESSED / 'dev.jsonl').read_text(encoding='utf-8').splitlines()]
    examples = [{k: next(r for r in dev if r['language'] == lang)[k] for k in ('sentence1', 'sentence2')} for lang in LANGUAGES]
    input_path = ROOT / 'data' / 'processed' / ('inference-input-' + RUN_ID + '.jsonl')
    input_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in examples), encoding='utf-8', newline='\n')
    result = predict(CHECKPOINT, input_path, ROOT / 'data' / 'processed' / ('inference-output-' + RUN_ID + '.json'))
    print(result)
else:
    print('Real encoder inference disabled by default; recorded proof is in docs when available.')


## 7. Final evidence and limitations
This notebook never calls final test evaluation. Release gold only through the explicit separate evaluation CLI after the serialized winner is frozen. Viewing already released aggregate reports does not create a new independent test. Source overlap, conditional component bootstrap, validation reuse and language-specific regressions must accompany any gain.

In [ ]:
final_reports = sorted((ROOT / 'docs').glob('final-test-v*.json'))
if final_reports:
    final = json.loads(final_reports[-1].read_text(encoding='utf-8'))
    print(final['test'])
    print(final['uncertainty'])
else:
    print('Final test gold remains sealed or no aggregate report has been published.')
